# 12 -- GPU utilization of the headline rows (plan v5 Step 11 support, Derek Nov 27)

Every comparison is at equal training GPU-hours, but a GPU-hour isn't equally busy in every arm (AlphaZero's
self-play measured ~0-2%). Per row: **utilization = sum of (phase hours x phase utilization) / total hours**.

1. **Split** (known, logs only): each run's hours by phase -- start (SFT), self-play, Stockfish (LLM RL, 0% GPU),
   update -- up to the row's round.
2. **Measure** (GPU): replay a slice of each arm with its real settings and checkpoint while a sampler reads the
   GPU's utilization twice a second. AlphaZero: one full round (~8 min each). LLM: the full game count, cut at
   `PLIES_*` moves. ~30-40 min in all on an L4. Saved after every arm to `results/gpu_util.json`; a rerun skips
   arms already measured (`REDO=1` to redo).
3. **Combine** -> one markdown line per row, to paste into the plan (Sec 4.0).

Assumptions (also in `eval/gpu_util.py`): the slice is representative (the last column compares seconds per
self-play position, slice / log; flagged if off by > 25%); LLM slices miss each round's small-batch tail; the
SFT parts of the LLM RL rows run at the 500k SFT run's utilization; `nvidia-smi` utilization = % of time any
kernel runs.


In [ ]:
# Setup (RunPod): the repo lives on the volume; artifacts under artifact_root (configs/base.yaml).
import os
_here = os.getcwd()
REPO_DIR = os.path.dirname(_here) if os.path.basename(_here) == "notebooks" else _here
%cd {REPO_DIR}
!git log -1 --oneline
!pip install -q -r requirements.txt

import json
import yaml
import torch
ROOT = yaml.safe_load(open("configs/base.yaml"))["artifact_root"]
from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- this needs a GPU pod"
print("GPU:", GPU)

In [ ]:
from eval import gpu_util as G

MEASURE = os.environ.get("MEASURE", "1") == "1"
REDO = os.environ.get("REDO", "0") == "1"
PLIES_LLM_RL = int(os.environ.get("PLIES_LLM_RL", "20"))
PLIES_LLM_MCTS = int(os.environ.get("PLIES_LLM_MCTS", "10"))
OUT = os.path.join(ROOT, "results", "gpu_util.json")

splits = {r.key: G.split(r, ROOT) for r in G.ROWS}
for r in G.ROWS:
    s = splits[r.key]
    print(f"{r.label:<40} " + ", ".join(f"{k} {v:.2f}" for k, v in s.items() if isinstance(v, float) and v)
          + f" | logged total {s.get('logged_total')}")


In [ ]:
import time as _time
measures = json.load(open(OUT)) if os.path.exists(OUT) and not REDO else {}
KW = {"llm_rl": {"plies": PLIES_LLM_RL}, "llm_mcts": {"plies": PLIES_LLM_MCTS}}
ORDER = ["az_scratch", "az_cnnsft", "sft500", "rl4", "selfplay_llm"]    # cheap first
if MEASURE:
    with G.GpuSampler(interval=0.5) as sampler:
        for key in ORDER:
            row = next(r for r in G.ROWS if r.key == key)
            if key in measures:
                print(f"{row.label}: already measured -- skipped"); continue
            t = _time.time()
            try:
                measures[key] = G.measure(row, ROOT, sampler, **KW.get(row.kind, {}))
            except Exception as e:      # one arm failing shouldn't lose the others
                print(f"{row.label}: FAILED -- {type(e).__name__}: {e}")
                import traceback; traceback.print_exc()
                continue
            json.dump(measures, open(OUT, "w"), indent=2)
            print(f"{row.label} ({_time.time() - t:.0f} s): {measures[key]}")
else:
    print("MEASURE off")


In [ ]:
combined = G.combine(splits, measures)
print(G.format_table(combined))
